In [1]:
import math
import numpy as np 
import netCDF4 as nc
import xarray as xr
import thermofeel as tmf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import geopandas as gpd
import rioxarray

import zipfile
import os
import shutil
import gc
import cdsapi
import glob
from datetime import datetime

In [3]:
os.chdir("../")
print(os.getcwd())

c:\Users\uginet\Documents\anemie\code_archive\replication_code_JEEM_anemia-main\0_input


In [4]:
def assign_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Summer"
    elif month in [6, 7, 8]:
        return "Kharif"
    elif month in [9, 10, 11]:
        return "Rabi"

### For data at resolution 010

In [5]:
# Collect all .nc file paths
nc_files = sorted(glob.glob(r"./ERA5_grid_cell_data/output/010_alltemps_*.nc"))

all_temps = {}
for f in nc_files:
    ds = xr.open_mfdataset(f, combine='by_coords').load()
    daily = ds.resample(time="1D").mean()
    all_temps[f]= daily
alltemps = xr.concat(all_temps.values(), dim='time')
print(alltemps)

<xarray.Dataset> Size: 63MB
Dimensions:  (time: 92, lat: 291, lon: 296)
Coordinates:
    number   int64 8B 0
  * lat      (lat) float64 2kB 35.5 35.4 35.3 35.2 35.1 ... 6.9 6.8 6.7 6.6 6.5
  * lon      (lon) float64 2kB 68.0 68.1 68.2 68.3 68.4 ... 97.2 97.3 97.4 97.5
  * time     (time) datetime64[ns] 736B 2014-10-01 2014-10-02 ... 2014-12-31
Data variables:
    t2m      (time, lat, lon) float32 32MB 282.7 284.3 288.1 ... nan nan nan
    tp       (time, lat, lon) float32 32MB 5.699e-07 5.699e-07 ... nan nan
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-03T10:45 GRIB to CDM+CF via cfgrib-0.9.1...


In [6]:
season = xr.DataArray(
    [assign_season(m) for m in alltemps['time'].dt.month.values],
    coords=[alltemps['time']],
    dims="time"
)

# Add season as a coordinate
alltemps = alltemps.assign_coords(season=season)

#compute season means
seasonal_means = alltemps.groupby('season').mean(dim='time')
print(seasonal_means)

#save season means as a new netcdf file
seasonal_means.to_netcdf('./avg_temp_seasons/output/010_avg_temp_per_season_x_pixel.nc')

<xarray.Dataset> Size: 1MB
Dimensions:  (season: 2, lat: 291, lon: 296)
Coordinates:
    number   int64 8B 0
  * lat      (lat) float64 2kB 35.5 35.4 35.3 35.2 35.1 ... 6.9 6.8 6.7 6.6 6.5
  * lon      (lon) float64 2kB 68.0 68.1 68.2 68.3 68.4 ... 97.2 97.3 97.4 97.5
  * season   (season) object 16B 'Rabi' 'Winter'
Data variables:
    t2m      (season, lat, lon) float32 689kB 274.0 275.5 278.7 ... nan nan nan
    tp       (season, lat, lon) float32 689kB 0.0003322 0.0003428 ... nan nan
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-10-03T10:45 GRIB to CDM+CF via cfgrib-0.9.1...


### For data at resolution 025

In [7]:
# Collect all .nc file paths
nc_files = sorted(glob.glob(r"./ERA5_grid_cell_data/output/025_alltemps_*.nc"))

all_temps = {}
for f in nc_files:
    ds = xr.open_mfdataset(f, combine='by_coords').load()
    daily = ds.resample(time="1D").mean()
    all_temps[f]= daily
alltemps = xr.concat(all_temps.values(), dim='time')
print(alltemps)

<xarray.Dataset> Size: 31MB
Dimensions:  (time: 61, lat: 117, lon: 119)
Coordinates:
    number   int64 8B 0
  * lat      (lat) float64 936B 35.5 35.25 35.0 34.75 34.5 ... 7.25 7.0 6.75 6.5
  * lon      (lon) float64 952B 68.0 68.25 68.5 68.75 ... 96.75 97.0 97.25 97.5
  * time     (time) datetime64[ns] 488B 2014-10-01 2014-10-02 ... 2014-11-30
Data variables:
    t2m      (time, lat, lon) float32 3MB 286.3 286.9 288.0 ... 300.7 300.9
    rh       (time, lat, lon) float32 3MB 37.06 35.64 31.17 ... 80.69 80.07
    wbt      (time, lat, lon) float32 3MB 279.3 279.6 279.9 ... 298.1 298.1
    mrt      (time, lat, lon) float32 3MB 288.7 290.0 290.2 ... 309.2 309.2
    bgt      (time, lat, lon) float64 7MB 287.8 288.3 289.4 ... 303.6 303.7
    wbgt     (time, lat, lon) float64 7MB 281.7 282.1 282.6 ... 299.4 299.5
    tp       (time, lat, lon) float32 3MB 0.0 0.0 0.0 ... 8.67e-05 7.127e-05
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Ra

In [8]:
season = xr.DataArray(
    [assign_season(m) for m in alltemps['time'].dt.month.values],
    coords=[alltemps['time']],
    dims="time"
)

# Add season as a coordinate
alltemps = alltemps.assign_coords(season=season)

#compute season means
seasonal_means = alltemps.groupby('season').mean(dim='time')
print(seasonal_means)

#save season means as a new netcdf file
seasonal_means.to_netcdf('./avg_temp_seasons/output/025_avg_temp_per_season_x_pixel.nc')

<xarray.Dataset> Size: 503kB
Dimensions:  (season: 1, lat: 117, lon: 119)
Coordinates:
    number   int64 8B 0
  * lat      (lat) float64 936B 35.5 35.25 35.0 34.75 34.5 ... 7.25 7.0 6.75 6.5
  * lon      (lon) float64 952B 68.0 68.25 68.5 68.75 ... 96.75 97.0 97.25 97.5
  * season   (season) object 8B 'Rabi'
Data variables:
    t2m      (season, lat, lon) float32 56kB 277.3 278.0 279.0 ... 300.2 300.2
    rh       (season, lat, lon) float32 56kB 58.78 60.61 56.54 ... 82.39 82.33
    wbt      (season, lat, lon) float32 56kB 273.7 274.4 275.0 ... 297.8 297.8
    mrt      (season, lat, lon) float32 56kB 280.3 281.5 281.8 ... 307.2 307.2
    bgt      (season, lat, lon) float64 111kB 278.8 279.5 280.5 ... 302.0 302.1
    wbgt     (season, lat, lon) float64 111kB 275.1 275.8 276.5 ... 298.9 298.9
    tp       (season, lat, lon) float32 56kB 2.579e-05 3.4e-05 ... 0.0003537
Attributes:
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Fore